In [5]:
import pandas as pd
import numpy as np
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import phik

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score

import joblib

In [6]:
classification_data = pd.read_csv(r'Data/financial_transactions.csv')
print(classification_data.info())
print(classification_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   amount           100000 non-null  float64
 1   weekday          100000 non-null  int64  
 2   month            100000 non-null  int64  
 3   description_len  100000 non-null  int64  
 4   is_income        100000 non-null  int64  
 5   category         100000 non-null  object 
dtypes: float64(1), int64(4), object(1)
memory usage: 4.6+ MB
None
     amount  weekday  month  description_len  is_income     category
0   4342.71        5      1               49          0  Развлечения
1  27619.44        5      7               33          1     Зарплата
2  14499.34        3      7               40          1     Зарплата
3   1302.40        4      4               15          0    Транспорт
4   4925.35        4      6               36          0    Транспорт


In [7]:
RANDOM_STATE = 42
y = classification_data['category']
X = classification_data.drop('category', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.1)

In [8]:
cat_cols = []
num_cols = [col for col in classification_data.select_dtypes(include=['int64', 'float64']).columns if col != 'category']

In [9]:
label_pipe = Pipeline([
    ('Impute', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
    ('label', LabelEncoder())
])

In [10]:
data_preprosessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
])

In [11]:
lr_pipe = Pipeline([
    ('preprosessor', data_preprosessor),
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE))
])

lr_auc = cross_val_score(lr_pipe, X, y, cv=5, scoring='roc_auc_ovo').mean()

l_start = time.time()
lr_pipe.fit(X_train, y_train)
l_finish = time.time()

p_start = time.time()
y_pred = lr_pipe.predict(X_test)
p_finish = time.time()

lr_stats = {
    'Время обучения': l_finish - l_start,
    'Время предсказания': p_finish - p_start,
    'ROC_AUC': lr_auc
}

for key, value in lr_stats.items():
    print(f'{key}: {value}')

Время обучения: 0.7981040477752686
Время предсказания: 0.018787622451782227
ROC_AUC: 0.7683334328188189


In [18]:
def find_preprosessing_score(model):
    results_df = pd.DataFrame(model.cv_results_)
    best_scores_df = results_df[results_df['rank_test_score'] == 1]

    model_stats = {
        'Время обучения': round(best_scores_df['mean_fit_time'].iloc[0], 3),
        'Время предсказания': round(best_scores_df['mean_score_time'].iloc[0], 3),
        'roc_auc': round(model.best_score_,3)
    }

    for key, value in model_stats.items():
        print(f'{key}: {value}')
    return model_stats

In [20]:
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear', 'saga'],  # l1 работает не со всеми солверами
    'model__class_weight': [None, 'balanced'],
    'model__max_iter': [100, 200, 500]
}

lr_grid = GridSearchCV(
    estimator=lr_pipe,
    param_grid=param_grid,
    scoring='roc_auc_ovo', 
    cv=5,
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)
lr_tune_stats = find_preprosessing_score(lr_grid)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


KeyboardInterrupt: 

In [15]:
rf_pipe = Pipeline([
    ('preprossesor', data_preprosessor),
    ('model', RandomForestClassifier())
])

rf_params_grid = {
    'model__n_estimators': np.arange(100, 1001, 100),    
    'model__max_depth': np.arange(3, 31, 3),                
    'model__min_samples_split': np.arange(2, 21, 2),       
    'model__min_samples_leaf': np.arange(1, 11, 2),        
    'model__max_features': np.linspace(0.3, 1.0, 8),        
    'model__bootstrap': [True, False]                       
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_params_grid,
    cv=5,
    scoring='roc_auc_ovo',
    n_jobs = -1
)

rf_search.fit(X_train, y_train)
rf_stats = find_preprosessing_score(rf_search)

Время обучения: 19.069
Время предсказания: 0.529
roc_auc: 0.768


In [16]:
lgbm_pipe = Pipeline([
    ('preprocessor', data_preprosessor),
    ('lgbm', LGBMClassifier())
])

lgbm_params = {
    'lgbm__n_estimators': [300, 500],
    'lgbm__learning_rate': [0.1, 0.3, 0.5]
}

X_train_lgbm = X_train.copy()
X_test_lgbm = X_test.copy()

lgbm_search = GridSearchCV(
    lgbm_pipe,
    lgbm_params,
    cv=5,
    scoring='roc_auc_ovo',
    n_jobs=-1
)

lgbm_search.fit(X_train_lgbm, y_train)
lgbm_stat = find_preprosessing_score(lgbm_search)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000854 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 326
[LightGBM] [Info] Number of data points in the train set: 90000, number of used features: 5
[LightGBM] [Info] Start training from score -1.560277
[LightGBM] [Info] Start training from score -2.246835
[LightGBM] [Info] Start training from score -1.716219
[LightGBM] [Info] Start training from score -3.533967
[LightGBM] [Info] Start training from score -2.404864
[LightGBM] [Info] Start training from score -3.361336
[LightGBM] [Info] Start training from score -1.752795
[LightGBM] [Info] Start training from score -1.731606
Время обучения: 86.888
Время предсказания: 2.167
roc_auc: 0.768


In [17]:
models = ['LogisticRegression', 'RandomForest', 'LGBM']
models_info = [lr_stats,rf_stats, lgbm_stat]

pd.DataFrame(models_info, index=models)

,Время обучения,Время предсказания,ROC_AUC,roc_auc
LogisticRegression,0.798104,0.018788,0.768333,NaN
RandomForest,19.069000,0.529000,NaN,0.768
LGBM,86.888000,2.167000,NaN,0.768


In [22]:
import os

# ensure the target directory exists and then save the model to a file inside it
os.makedirs('Ready_models', exist_ok=True)
joblib.dump(lr_pipe, 'Ready_models/lr_pipe.joblib')

['Ready_models/lr_pipe.joblib']